In [3]:
from pathlib import Path
import os
import sys
import json
import time

import numpy as np
import pandas as pd
import lightgbm as lgb

REPO = Path.home() / "Amazon-ML-Challenge"
CODE_ROOT = REPO / "code" / "business_entity_resolution"
SRC = CODE_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

os.chdir(REPO)

print("Repo:", REPO)
print("Python:", sys.version.split()[0])
print("LightGBM:", lgb.__version__)
print("Working directory:", Path.cwd())

Repo: /home/jupyter-1nt23cb058/Amazon-ML-Challenge
Python: 3.9.7
LightGBM: 4.6.0
Working directory: /home/jupyter-1nt23cb058/Amazon-ML-Challenge


In [4]:
from collections import defaultdict

search_roots = [
    REPO / "cache",
    REPO / "models",
    REPO / "experiments",
]

patterns = ["*.parquet", "*.csv", "*.tsv", "*.npy", "*.npz", "*.json", "*.txt"]

found = []

for root in search_roots:
    if not root.exists():
        continue

    for pattern in patterns:
        for p in root.rglob(pattern):
            found.append(
                (
                    str(p.relative_to(REPO)),
                    p.stat().st_size / (1024**2),
                    p.stat().st_mtime,
                )
            )

found = sorted(found, key=lambda x: x[2], reverse=True)

print(f"Found {len(found)} artifacts\n")

for path, size_mb, mtime in found[:100]:
    print(f"{size_mb:9.2f} MB   {path}")

Found 64 artifacts

     0.07 MB   experiments/e23_retrieval_38a0b640f78b_metrics.json
     0.00 MB   cache/candidates/e23_retrieval_38a0b640f78b_selected.parquet.meta.json
    10.22 MB   cache/candidates/e23_retrieval_38a0b640f78b_selected.parquet
     0.00 MB   cache/candidates/e23_retrieval_38a0b640f78b_address_tfidf.parquet.meta.json
     3.19 MB   cache/candidates/e23_retrieval_38a0b640f78b_address_tfidf.parquet
     0.07 MB   experiments/e23_retrieval_bb2d7121ad1c_metrics.json
    10.22 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_selected.parquet
     0.00 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_selected.parquet.meta.json
     3.19 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_address_tfidf.parquet
     0.00 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_address_tfidf.parquet.meta.json
     0.10 MB   experiments/e22_retrieval_e7c71a736237_metrics.json
     0.00 MB   cache/candidates/e22_retrieval_e7c71a736237_selected.parquet.meta.json
     9.80 MB   cache/cand

In [5]:
keywords = [
    "e2",
    "e21",
    "e22",
    "e23",
    "candidate",
    "feature",
    "prediction",
    "validation",
    "train",
]

interesting = []

for path, size_mb, mtime in found:
    lower = path.lower()
    if any(k in lower for k in keywords):
        interesting.append((path, size_mb))

for path, size_mb in interesting[:100]:
    print(f"{size_mb:9.2f} MB   {path}")

     0.07 MB   experiments/e23_retrieval_38a0b640f78b_metrics.json
     0.00 MB   cache/candidates/e23_retrieval_38a0b640f78b_selected.parquet.meta.json
    10.22 MB   cache/candidates/e23_retrieval_38a0b640f78b_selected.parquet
     0.00 MB   cache/candidates/e23_retrieval_38a0b640f78b_address_tfidf.parquet.meta.json
     3.19 MB   cache/candidates/e23_retrieval_38a0b640f78b_address_tfidf.parquet
     0.07 MB   experiments/e23_retrieval_bb2d7121ad1c_metrics.json
    10.22 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_selected.parquet
     0.00 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_selected.parquet.meta.json
     3.19 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_address_tfidf.parquet
     0.00 MB   cache/candidates/e23_retrieval_bb2d7121ad1c_address_tfidf.parquet.meta.json
     0.10 MB   experiments/e22_retrieval_e7c71a736237_metrics.json
     0.00 MB   cache/candidates/e22_retrieval_e7c71a736237_selected.parquet.meta.json
     9.80 MB   cache/candidates/e22_retrieval

In [6]:
from pathlib import Path
import json
import pprint

E23_RUNS = [
    "e23_retrieval_38a0b640f78b",
    "e23_retrieval_bb2d7121ad1c",
]

for run in E23_RUNS:
    path = REPO / "experiments" / f"{run}_metrics.json"
    print("\n" + "=" * 100)
    print(run)
    print("=" * 100)

    with open(path) as f:
        metrics = json.load(f)

    pprint.pp(metrics, depth=4, width=120)


e23_retrieval_38a0b640f78b
{'artifact_id': '38a0b640f78b',
 'artifacts': {'optimized_address_candidates': '/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/candidates/e23_retrieval_38a0b640f78b_address_tfidf.parquet',
               'report': '/home/jupyter-1nt23cb058/Amazon-ML-Challenge/experiments/e23_retrieval_38a0b640f78b_metrics.json',
               'selected_candidates': '/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/candidates/e23_retrieval_38a0b640f78b_selected.parquet'},
 'cache_disk_usage_bytes': {'new_candidate_caches_total': 14060027,
                            'optimized_address_candidates': 3347642,
                            'selected_candidates': 10712385,
                            'sparse_index_cached': False},
 'cache_reuse': {'e21_candidates': '/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/candidates/e21_lgbm_c449c0c6b281_candidates.parquet',
                 'feed_rows': 1034502,
                 'legacy_candidates': '/home/jupyter-1nt23cb058/Amazo

                                                  {'address_tfidf_rank': 9,
                                                   'address_tfidf_similarity': 0.7342166900634766,
                                                   'candidate_address': 'Sacramento, 07916 Valley Green Dr, California',
                                                   'candidate_entity_id': 'S3-638659610',
                                                   'candidate_name': 'Bexi Lng',
                                                   'country': 'us',
                                                   'failure_category': 'other',
                                                   'routes': {},
                                                   'source1_address': '7916 Valley Green Drive, Sacramento, CA',
                                                   'source1_entity_id': 'S1-369212908',
                                                   'source1_name': 'Beix Lng'},
                                       

In [8]:
for run in E23_RUNS:
    path = REPO / "cache" / "candidates" / f"{run}_selected.parquet"
    df = pd.read_parquet(path)

    print("\n", run)
    print("shape:", df.shape)
    print("columns:")
    print(df.columns.tolist())
    display(df.head(3))


 e23_retrieval_38a0b640f78b
shape: (488090, 45)
columns:
['source1_entity_id', 'candidate_entity_id', 'candidate_source', 'same_country_block', 'cheap_score', 'cheap_rank', 'cheap_score_margin_to_next', 'rare_name_token_hits', 'rare_address_token_hits', 'block_exact_full_name', 'block_exact_core_name', 'block_exact_sorted_name', 'block_rare_name_token', 'block_exact_address', 'block_rare_address_token', 'block_cross_country_exact_name', 'has_legacy', 'name_tfidf_similarity', 'name_tfidf_rank', 'has_tfidf', 'legacy_rank', 'legacy_cheap_score', 'block_name_tfidf', 'is_tfidf_only', 'is_legacy_only', 'supported_by_both', 'legacy_blocking_channel_count', 'candidate_count_raw', 'address_tfidf_similarity', 'address_tfidf_rank', 'address_route_postcode', 'address_route_postcode_prefix', 'address_route_numeric', 'address_route_rare_token', 'address_route_country_fallback', 'address_route_count', 'address_preblock_pool_size', 'has_address_tfidf', 'block_address_tfidf', 'is_address_tfidf_only', 

,source1_entity_id,candidate_entity_id,candidate_source,same_country_block,cheap_score,cheap_rank,cheap_score_margin_to_next,rare_name_token_hits,rare_address_token_hits,block_exact_full_name,...,address_route_count,address_preblock_pool_size,has_address_tfidf,block_address_tfidf,is_address_tfidf_only,retrieval_channel_count,fusion_score,fusion_rank,candidate_count_for_s1,fusion_strategy
0,S1-100077788,S3-490363076,S3,1,15.0,1,5.0,1.0,0.0,1.0,...,1,177,1,1,0,3,0.190476,1,50,protected35_lw2_nw1_aw1_c20
1,S1-100077788,S3-771950070,S3,1,10.0,2,8.5,1.0,0.0,0.0,...,1,177,1,1,0,3,0.181818,2,50,protected35_lw2_nw1_aw1_c20
2,S1-100077788,S2-454666788,S2,1,1.5,3,0.0,1.0,0.0,0.0,...,0,0,0,0,0,2,0.120401,3,50,protected35_lw2_nw1_aw1_c20



 e23_retrieval_bb2d7121ad1c
shape: (488091, 45)
columns:
['source1_entity_id', 'candidate_entity_id', 'candidate_source', 'same_country_block', 'cheap_score', 'cheap_rank', 'cheap_score_margin_to_next', 'rare_name_token_hits', 'rare_address_token_hits', 'block_exact_full_name', 'block_exact_core_name', 'block_exact_sorted_name', 'block_rare_name_token', 'block_exact_address', 'block_rare_address_token', 'block_cross_country_exact_name', 'has_legacy', 'name_tfidf_similarity', 'name_tfidf_rank', 'has_tfidf', 'legacy_rank', 'legacy_cheap_score', 'block_name_tfidf', 'is_tfidf_only', 'is_legacy_only', 'supported_by_both', 'legacy_blocking_channel_count', 'candidate_count_raw', 'address_tfidf_similarity', 'address_tfidf_rank', 'address_route_postcode', 'address_route_postcode_prefix', 'address_route_numeric', 'address_route_rare_token', 'address_route_country_fallback', 'address_route_count', 'address_preblock_pool_size', 'has_address_tfidf', 'block_address_tfidf', 'is_address_tfidf_only', 

,source1_entity_id,candidate_entity_id,candidate_source,same_country_block,cheap_score,cheap_rank,cheap_score_margin_to_next,rare_name_token_hits,rare_address_token_hits,block_exact_full_name,...,address_route_count,address_preblock_pool_size,has_address_tfidf,block_address_tfidf,is_address_tfidf_only,retrieval_channel_count,fusion_score,fusion_rank,candidate_count_for_s1,fusion_strategy
0,S1-100077788,S3-490363076,S3,1,15.0,1,5.0,1.0,0.0,1.0,...,1,177,1,1,0,3,0.190476,1,50,protected35_lw2_nw1_aw1_c20
1,S1-100077788,S3-771950070,S3,1,10.0,2,8.5,1.0,0.0,0.0,...,1,177,1,1,0,3,0.181818,2,50,protected35_lw2_nw1_aw1_c20
2,S1-100077788,S2-454666788,S2,1,1.5,3,0.0,1.0,0.0,0.0,...,0,0,0,0,0,2,0.120401,3,50,protected35_lw2_nw1_aw1_c20


In [9]:
OLD_FEATURE_PATH = (
    REPO
    / "cache"
    / "features"
    / "e21_lgbm_c449c0c6b281_features.parquet"
)

old_features = pd.read_parquet(OLD_FEATURE_PATH)

print("Old E2.1 feature table:", old_features.shape)
print("\nColumns:")
for i, col in enumerate(old_features.columns):
    print(f"{i:02d}: {col}")

display(old_features.head(3))

Old E2.1 feature table: (403984, 63)

Columns:
00: source1_entity_id
01: candidate_entity_id
02: name_full_exact
03: name_core_exact
04: name_sorted_exact
05: name_token_jaccard
06: name_shared_token_count
07: name_token_count_difference
08: name_length_ratio
09: name_character_similarity
10: name_prefix_ratio
11: name_suffix_ratio
12: address_exact
13: address_token_jaccard
14: address_shared_token_count
15: numeric_token_equal
16: numeric_token_overlap
17: numeric_token_conflict
18: address_length_ratio
19: address_missing_left
20: address_missing_right
21: same_country
22: country_conflict
23: country_missing_left
24: country_missing_right
25: both_country_present
26: source_is_s2
27: source_is_s3
28: block_exact_full_name
29: block_exact_core_name
30: block_exact_sorted_name
31: block_rare_name_token
32: block_exact_address
33: block_rare_address_token
34: block_cross_country_exact_name
35: block_name_tfidf
36: blocking_channel_count
37: rare_name_token_hits
38: rare_address_token_

,source1_entity_id,candidate_entity_id,name_full_exact,name_core_exact,name_sorted_exact,name_token_jaccard,name_shared_token_count,name_token_count_difference,name_length_ratio,name_character_similarity,...,label,relative_name_similarity_rank,relative_address_similarity_rank,relative_tfidf_similarity_rank,relative_fusion_score_rank,top_name_similarity_margin,top_address_similarity_margin,top_tfidf_similarity_margin,fusion_score_gap_to_best,close_name_competitor_count
0,S1-100077788,S3-490363076,1,1,1,1.00,4,0,1.000000,1.000000,...,1,1,2,1,1,0.020408,0.036364,0.0,0.000000,2
1,S1-100077788,S3-771950070,0,1,1,0.75,3,1,0.833333,0.909091,...,1,3,1,2,2,0.020408,0.036364,0.0,0.006494,2
2,S1-100077788,S2-454666788,0,0,0,0.60,3,0,0.960000,0.979592,...,0,2,14,3,3,0.020408,0.036364,0.0,0.022456,2


In [10]:
with open(
    REPO / "models" / "lgbm" / "e21_lgbm_c449c0c6b281_features.json"
) as f:
    old_feature_spec = json.load(f)

old_feature_spec

['name_full_exact',
 'name_core_exact',
 'name_sorted_exact',
 'name_token_jaccard',
 'name_shared_token_count',
 'name_token_count_difference',
 'name_length_ratio',
 'name_character_similarity',
 'name_prefix_ratio',
 'name_suffix_ratio',
 'address_exact',
 'address_token_jaccard',
 'address_shared_token_count',
 'numeric_token_equal',
 'numeric_token_overlap',
 'numeric_token_conflict',
 'address_length_ratio',
 'address_missing_left',
 'address_missing_right',
 'same_country',
 'country_conflict',
 'country_missing_left',
 'country_missing_right',
 'both_country_present',
 'source_is_s2',
 'source_is_s3',
 'block_exact_full_name',
 'block_exact_core_name',
 'block_exact_sorted_name',
 'block_rare_name_token',
 'block_exact_address',
 'block_rare_address_token',
 'block_cross_country_exact_name',
 'block_name_tfidf',
 'blocking_channel_count',
 'rare_name_token_hits',
 'rare_address_token_hits',
 'name_tfidf_similarity',
 'name_tfidf_rank',
 'legacy_blocking_channel_count',
 'legacy

In [11]:
import inspect

import business_entity_resolution.features as feature_module

print(inspect.getsource(feature_module)[:30000])

"""Compact, LightGBM-ready numeric features for blocked candidate pairs."""

from __future__ import annotations

from difflib import SequenceMatcher
from typing import Dict, Iterator, List, Set

import numpy as np
import pandas as pd

from .blocking import PROVENANCE_COLUMNS


IDENTIFIER_COLUMNS = ["source1_entity_id", "candidate_entity_id"]

FEATURE_COLUMNS = [
    "name_full_exact",
    "name_core_exact",
    "name_sorted_exact",
    "name_token_jaccard",
    "name_shared_token_count",
    "name_token_count_difference",
    "name_length_ratio",
    "name_character_similarity",
    "name_prefix_ratio",
    "name_suffix_ratio",
    "address_exact",
    "address_token_jaccard",
    "address_shared_token_count",
    "numeric_token_equal",
    "numeric_token_overlap",
    "numeric_token_conflict",
    "address_length_ratio",
    "address_missing_left",
    "address_missing_right",
    "same_country",
    "country_conflict",
    "country_missing_left",
    "country_missing_right",
    "bot

In [12]:
# Cell 9: Load the selected E2.3 candidate set and the normalized Source1/feed records
# that we will use to build pairwise model features.

RUN = "e23_retrieval_38a0b640f78b"

CANDIDATE_PATH = (
    REPO / "cache" / "candidates" / f"{RUN}_selected.parquet"
)

SOURCE1_PATH = (
    REPO
    / "cache"
    / "normalized"
    / "e21_lgbm_c449c0c6b281_source1.parquet"
)

FEED_PATH = (
    REPO
    / "cache"
    / "normalized"
    / "e21_lgbm_c449c0c6b281_feed.parquet"
)

candidates = pd.read_parquet(CANDIDATE_PATH)
source1 = pd.read_parquet(SOURCE1_PATH)
feed = pd.read_parquet(FEED_PATH)

print("Candidates:", candidates.shape)
print("Source1:", source1.shape)
print("Feed:", feed.shape)

assert len(candidates) == 488090
assert candidates["source1_entity_id"].nunique() == 10_000

print("✅ E2.3 data loaded")

Candidates: (488090, 45)
Source1: (10000, 18)
Feed: (1034502, 23)
✅ E2.3 data loaded


In [13]:
# Cell 10: Build the base pairwise feature table for all E2.3 candidate pairs
# using the existing feature-engineering pipeline. This is CPU feature generation,
# not model training yet.

from business_entity_resolution.features import build_feature_table

t0 = time.time()

features = build_feature_table(
    candidates,
    source1,
    feed,
    chunk_size=50_000,
)

elapsed = time.time() - t0

print(f"Base features generated in {elapsed:.1f}s")
print("Feature table shape:", features.shape)
print("Feature columns:", len(features.columns))

display(features.head(3))

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [14]:
# Cell 10A: Identify which candidate columns contain NaN/inf values that break
# the old feature builder's integer downcasting. This cell only inspects data.

problem_cols = []

for col in candidates.columns:
    if pd.api.types.is_numeric_dtype(candidates[col]):
        n_nan = candidates[col].isna().sum()
        n_inf = np.isinf(
            pd.to_numeric(candidates[col], errors="coerce").fillna(0)
        ).sum()

        if n_nan > 0 or n_inf > 0:
            problem_cols.append((col, int(n_nan), int(n_inf)))

print("Columns with NaN/inf:")
for col, n_nan, n_inf in problem_cols:
    print(f"{col:40s} NaN={n_nan:8d}  inf={n_inf:8d}")

print("\nTotal problematic columns:", len(problem_cols))

Columns with NaN/inf:
cheap_score                              NaN=  111314  inf=       0
cheap_score_margin_to_next               NaN=  111314  inf=       0
rare_name_token_hits                     NaN=  111314  inf=       0
rare_address_token_hits                  NaN=  111314  inf=       0
block_exact_full_name                    NaN=  111314  inf=       0
block_exact_core_name                    NaN=  111314  inf=       0
block_exact_sorted_name                  NaN=  111314  inf=       0
block_rare_name_token                    NaN=  111314  inf=       0
block_exact_address                      NaN=  111314  inf=       0
block_rare_address_token                 NaN=  111314  inf=       0
block_cross_country_exact_name           NaN=  111314  inf=       0
legacy_cheap_score                       NaN=  111314  inf=       0
block_name_tfidf                         NaN=  111314  inf=       0
is_tfidf_only                            NaN=  111314  inf=       0
is_legacy_only            

In [15]:
# Cell 10B: Fill missing legacy/name-retrieval metadata with neutral defaults
# for E2.3 address-only candidates so the old feature builder can process them.

candidates_clean = candidates.copy()

ZERO_FILL_COLS = [
    "cheap_score",
    "cheap_score_margin_to_next",
    "rare_name_token_hits",
    "rare_address_token_hits",
    "block_exact_full_name",
    "block_exact_core_name",
    "block_exact_sorted_name",
    "block_rare_name_token",
    "block_exact_address",
    "block_rare_address_token",
    "block_cross_country_exact_name",
    "legacy_cheap_score",
    "block_name_tfidf",
    "is_tfidf_only",
    "is_legacy_only",
    "supported_by_both",
    "legacy_blocking_channel_count",
    "candidate_count_raw",
]

for col in ZERO_FILL_COLS:
    candidates_clean[col] = candidates_clean[col].fillna(0)

print("Remaining NaNs in cleaned candidate table:")
remaining = candidates_clean[ZERO_FILL_COLS].isna().sum()
print(remaining[remaining > 0])

print("\nRows cleaned:", (candidates[ZERO_FILL_COLS].isna().any(axis=1)).sum())
print("✅ Legacy/name metadata cleaned")

Remaining NaNs in cleaned candidate table:
Series([], dtype: int64)

Rows cleaned: 111314
✅ Legacy/name metadata cleaned


In [16]:
# Cell 10C: Rebuild the base pairwise feature table using the cleaned E2.3
# candidate metadata. This is still feature engineering, not model training.

import time

t0 = time.time()

features = build_feature_table(
    candidates_clean,
    source1,
    feed,
    chunk_size=50_000,
)

elapsed = time.time() - t0

print(f"Base features generated in {elapsed:.1f}s")
print("Feature table shape:", features.shape)
print("Feature columns:", len(features.columns))

display(features.head(3))

Base features generated in 229.0s
Feature table shape: (488090, 53)
Feature columns: 53


,source1_entity_id,candidate_entity_id,name_full_exact,name_core_exact,name_sorted_exact,name_token_jaccard,name_shared_token_count,name_token_count_difference,name_length_ratio,name_character_similarity,...,legacy_cheap_score,is_tfidf_only,is_legacy_only,supported_by_both,fusion_score,fusion_rank,candidate_count_for_s1,cheap_score,cheap_rank,cheap_score_margin_to_next
0,S1-100077788,S3-490363076,1,1,1,1.00,4,0,1.000000,1.000000,...,15.0,0,0,1,0.190476,1,50,15.0,1,5.0
1,S1-100077788,S3-771950070,0,1,1,0.75,3,1,0.833333,0.909091,...,10.0,0,0,1,0.181818,2,50,10.0,2,8.5
2,S1-100077788,S2-454666788,0,0,0,0.60,3,0,0.960000,0.979592,...,1.5,0,0,1,0.120401,3,50,1.5,3,0.0


In [17]:
# Cell 11: Append E2.3 address-retrieval metadata to the base pairwise feature table.
# These are retrieval/provenance signals that were not part of the old E2.1 feature builder.

E23_EXTRA_COLUMNS = [
    "source1_entity_id",
    "candidate_entity_id",
    "address_tfidf_similarity",
    "address_tfidf_rank",
    "address_route_postcode",
    "address_route_postcode_prefix",
    "address_route_numeric",
    "address_route_rare_token",
    "address_route_country_fallback",
    "address_route_count",
    "address_preblock_pool_size",
    "has_address_tfidf",
    "block_address_tfidf",
    "is_address_tfidf_only",
    "retrieval_channel_count",
]

extra = candidates_clean[E23_EXTRA_COLUMNS].copy()

features = features.merge(
    extra,
    on=["source1_entity_id", "candidate_entity_id"],
    how="left",
    validate="one_to_one",
)

# Compact dtypes to keep memory under control.
binary_cols = [
    "address_route_postcode",
    "address_route_postcode_prefix",
    "address_route_numeric",
    "address_route_rare_token",
    "address_route_country_fallback",
    "has_address_tfidf",
    "block_address_tfidf",
    "is_address_tfidf_only",
]

for col in binary_cols:
    features[col] = features[col].fillna(0).astype("uint8")

features["address_tfidf_similarity"] = (
    features["address_tfidf_similarity"].fillna(0).astype("float32")
)

count_cols = [
    "address_tfidf_rank",
    "address_route_count",
    "address_preblock_pool_size",
    "retrieval_channel_count",
]

for col in count_cols:
    features[col] = features[col].fillna(0).astype("uint32")

print("Feature table shape:", features.shape)
print("Columns:", len(features.columns))
print("Remaining NaNs in E2.3 extras:",
      int(features[E23_EXTRA_COLUMNS[2:]].isna().sum().sum()))

display(
    features[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "address_tfidf_similarity",
            "address_tfidf_rank",
            "has_address_tfidf",
            "is_address_tfidf_only",
            "retrieval_channel_count",
        ]
    ].head(5)
)

Feature table shape: (488090, 66)
Columns: 66
Remaining NaNs in E2.3 extras: 0


,source1_entity_id,candidate_entity_id,address_tfidf_similarity,address_tfidf_rank,has_address_tfidf,is_address_tfidf_only,retrieval_channel_count
0,S1-100077788,S3-490363076,0.725058,1,1,0,3
1,S1-100077788,S3-771950070,0.713299,2,1,0,3
2,S1-100077788,S2-454666788,0.000000,0,0,0,2
3,S1-100077788,S3-291121444,0.000000,0,0,0,2
4,S1-100077788,S2-204267984,0.000000,0,0,0,1


In [18]:
# Cell 12: Add the existing candidate-relative features such as within-entity
# similarity ranks and score margins used by the previous LightGBM model.

from business_entity_resolution.features import add_candidate_relative_features

t0 = time.time()

features = add_candidate_relative_features(features)

elapsed = time.time() - t0

print(f"Relative features added in {elapsed:.1f}s")
print("Feature table shape:", features.shape)
print("Columns:", len(features.columns))

display(
    features[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "relative_name_similarity_rank",
            "relative_address_similarity_rank",
            "relative_tfidf_similarity_rank",
            "relative_fusion_score_rank",
            "top_name_similarity_margin",
            "top_address_similarity_margin",
            "top_tfidf_similarity_margin",
            "fusion_score_gap_to_best",
            "close_name_competitor_count",
        ]
    ].head(5)
)

Relative features added in 5.0s
Feature table shape: (488090, 75)
Columns: 75


,source1_entity_id,candidate_entity_id,relative_name_similarity_rank,relative_address_similarity_rank,relative_tfidf_similarity_rank,relative_fusion_score_rank,top_name_similarity_margin,top_address_similarity_margin,top_tfidf_similarity_margin,fusion_score_gap_to_best,close_name_competitor_count
0,S1-100077788,S3-490363076,1,4,1,1,0.020408,0.036364,0.0,0.000000,2
1,S1-100077788,S3-771950070,3,1,2,2,0.020408,0.036364,0.0,0.008658,2
2,S1-100077788,S2-454666788,2,27,3,3,0.020408,0.036364,0.0,0.070075,2
3,S1-100077788,S3-291121444,23,38,5,4,0.020408,0.036364,0.0,0.096422,2
4,S1-100077788,S2-204267984,31,21,22,5,0.020408,0.036364,0.0,0.103520,2


In [19]:
# Cell 13: Add within-entity relative features for the new address-TFIDF signal,
# so LightGBM can use both absolute similarity and how strong it is versus competitors.

from business_entity_resolution.features import _deterministic_descending_rank

features["relative_address_tfidf_rank"] = (
    _deterministic_descending_rank(
        features,
        "address_tfidf_similarity",
    ).astype("uint32")
)

best_address_tfidf = (
    features
    .groupby("source1_entity_id")["address_tfidf_similarity"]
    .transform("max")
)

features["address_tfidf_gap_to_best"] = (
    best_address_tfidf - features["address_tfidf_similarity"]
).astype("float32")

print("Feature table shape:", features.shape)
print("Columns:", len(features.columns))

display(
    features[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "address_tfidf_similarity",
            "relative_address_tfidf_rank",
            "address_tfidf_gap_to_best",
        ]
    ].head(10)
)

Feature table shape: (488090, 77)
Columns: 77


,source1_entity_id,candidate_entity_id,address_tfidf_similarity,relative_address_tfidf_rank,address_tfidf_gap_to_best
0,S1-100077788,S3-490363076,0.725058,1,0.000000
1,S1-100077788,S3-771950070,0.713299,2,0.011759
2,S1-100077788,S2-454666788,0.000000,24,0.725058
3,S1-100077788,S3-291121444,0.000000,37,0.725058
4,S1-100077788,S2-204267984,0.000000,18,0.725058
5,S1-100077788,S2-233535286,0.000000,19,0.725058
6,S1-100077788,S2-326312152,0.000000,20,0.725058
7,S1-100077788,S2-634900154,0.000000,25,0.725058
8,S1-100077788,S3-408233518,0.000000,39,0.725058
9,S1-100077788,S3-433788926,0.000000,40,0.725058


In [20]:
# Cell 14: Locate the training ground-truth file and inspect its columns/schema
# before converting true entity matches into binary candidate-pair labels.

from pathlib import Path

gt_candidates = list(
    (REPO / "student_resource").rglob("*ground*truth*.tsv")
)

print("Ground-truth files found:")
for p in gt_candidates:
    print(" -", p)

assert gt_candidates, "No ground-truth TSV found under student_resource"

GT_PATH = gt_candidates[0]

gt = pd.read_csv(GT_PATH, sep="\t")

print("\nUsing:", GT_PATH)
print("Shape:", gt.shape)
print("Columns:", gt.columns.tolist())

display(gt.head(5))

Ground-truth files found:
 - /home/jupyter-1nt23cb058/Amazon-ML-Challenge/student_resource/dataset/train/train_ground_truth.tsv

Using: /home/jupyter-1nt23cb058/Amazon-ML-Challenge/student_resource/dataset/train/train_ground_truth.tsv
Shape: (2206821, 2)
Columns: ['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [21]:
# Cell 15: Build binary labels for the 488,090 candidate pairs by filtering the
# ground truth to our 10,000 Source1 entities, exploding true matches, and joining.

experiment_s1_ids = set(features["source1_entity_id"].unique())

gt_subset = gt[
    gt["source1_entity_id"].isin(experiment_s1_ids)
].copy()

# Turn comma-separated matched IDs into one true pair per row.
true_pairs = (
    gt_subset
    .assign(
        candidate_entity_id=
        gt_subset["matched_entity_ids"]
        .fillna("")
        .astype(str)
        .str.split(",")
    )
    .explode("candidate_entity_id")
)

# Remove empty-match rows.
true_pairs["candidate_entity_id"] = (
    true_pairs["candidate_entity_id"].astype(str).str.strip()
)

true_pairs = true_pairs[
    true_pairs["candidate_entity_id"] != ""
][["source1_entity_id", "candidate_entity_id"]].drop_duplicates()

true_pairs["label"] = 1

# Join labels onto candidate feature table; all non-true candidates become negatives.
features = features.merge(
    true_pairs,
    on=["source1_entity_id", "candidate_entity_id"],
    how="left",
    validate="one_to_one",
)

features["label"] = features["label"].fillna(0).astype("uint8")

print("Experiment Source1 entities:", len(experiment_s1_ids))
print("True links in ground truth subset:", len(true_pairs))
print("Positive candidate pairs recovered:", int(features["label"].sum()))
print("Negative candidate pairs:", int((features["label"] == 0).sum()))
print("Positive rate:", f"{features['label'].mean():.4%}")
print("Feature table shape:", features.shape)

assert features["source1_entity_id"].nunique() == 10_000
assert len(features) == 488_090

print("✅ Labels attached")

Experiment Source1 entities: 10000
True links in ground truth subset: 34502
Positive candidate pairs recovered: 33667
Negative candidate pairs: 454423
Positive rate: 6.8977%
Feature table shape: (488090, 78)
✅ Labels attached


In [22]:
# Cell 16: Recreate the exact E2.1/E2.3 train/validation split by loading the
# previously saved validation predictions and extracting their Source1 entity IDs.

OLD_VAL_PRED_PATH = (
    REPO
    / "cache"
    / "predictions"
    / "e21_lgbm_c449c0c6b281_validation.parquet"
)

old_val_predictions = pd.read_parquet(OLD_VAL_PRED_PATH)

val_s1_ids = set(
    old_val_predictions["source1_entity_id"].unique()
)

all_s1_ids = set(
    features["source1_entity_id"].unique()
)

train_s1_ids = all_s1_ids - val_s1_ids

print("Train Source1 entities:", len(train_s1_ids))
print("Validation Source1 entities:", len(val_s1_ids))
print("Total:", len(train_s1_ids) + len(val_s1_ids))

assert len(train_s1_ids) == 7568
assert len(val_s1_ids) == 2432
assert train_s1_ids.isdisjoint(val_s1_ids)

print("✅ Exact previous entity split recovered")

Train Source1 entities: 7568
Validation Source1 entities: 2432
Total: 10000
✅ Exact previous entity split recovered


In [23]:
# Cell 17: Build the final LightGBM feature list and create train/validation
# matrices using the exact entity-level split. This prepares training only.

OLD_MODEL_FEATURES = [
    "name_full_exact",
    "name_core_exact",
    "name_sorted_exact",
    "name_token_jaccard",
    "name_shared_token_count",
    "name_token_count_difference",
    "name_length_ratio",
    "name_character_similarity",
    "name_prefix_ratio",
    "name_suffix_ratio",
    "address_exact",
    "address_token_jaccard",
    "address_shared_token_count",
    "numeric_token_equal",
    "numeric_token_overlap",
    "numeric_token_conflict",
    "address_length_ratio",
    "address_missing_left",
    "address_missing_right",
    "same_country",
    "country_conflict",
    "country_missing_left",
    "country_missing_right",
    "both_country_present",
    "source_is_s2",
    "source_is_s3",
    "block_exact_full_name",
    "block_exact_core_name",
    "block_exact_sorted_name",
    "block_rare_name_token",
    "block_exact_address",
    "block_rare_address_token",
    "block_cross_country_exact_name",
    "block_name_tfidf",
    "blocking_channel_count",
    "rare_name_token_hits",
    "rare_address_token_hits",
    "name_tfidf_similarity",
    "name_tfidf_rank",
    "legacy_blocking_channel_count",
    "legacy_rank",
    "legacy_cheap_score",
    "is_tfidf_only",
    "is_legacy_only",
    "supported_by_both",
    "fusion_score",
    "fusion_rank",
    "candidate_count_for_s1",
    "cheap_score",
    "cheap_rank",
    "cheap_score_margin_to_next",
    "relative_name_similarity_rank",
    "relative_address_similarity_rank",
    "relative_tfidf_similarity_rank",
    "relative_fusion_score_rank",
    "top_name_similarity_margin",
    "top_address_similarity_margin",
    "top_tfidf_similarity_margin",
    "fusion_score_gap_to_best",
    "close_name_competitor_count",
]

E23_MODEL_FEATURES = [
    "address_tfidf_similarity",
    "address_tfidf_rank",
    "address_route_postcode",
    "address_route_postcode_prefix",
    "address_route_numeric",
    "address_route_rare_token",
    "address_route_country_fallback",
    "address_route_count",
    "address_preblock_pool_size",
    "has_address_tfidf",
    "block_address_tfidf",
    "is_address_tfidf_only",
    "retrieval_channel_count",
    "relative_address_tfidf_rank",
    "address_tfidf_gap_to_best",
]

model_features = OLD_MODEL_FEATURES + E23_MODEL_FEATURES

missing = [c for c in model_features if c not in features.columns]
assert not missing, f"Missing model features: {missing}"

train_mask = features["source1_entity_id"].isin(train_s1_ids)
val_mask = features["source1_entity_id"].isin(val_s1_ids)

X_train = features.loc[train_mask, model_features]
y_train = features.loc[train_mask, "label"]

X_val = features.loc[val_mask, model_features]
y_val = features.loc[val_mask, "label"]

print("Model features:", len(model_features))
print("Train pairs:", len(X_train))
print("Validation pairs:", len(X_val))
print("Train positives:", int(y_train.sum()))
print("Validation positives:", int(y_val.sum()))
print("Train positive rate:", f"{y_train.mean():.4%}")
print("Validation positive rate:", f"{y_val.mean():.4%}")

assert X_train.shape[1] == 75
assert X_val.shape[1] == 75

print("✅ LightGBM matrices ready")

Model features: 75
Train pairs: 369671
Validation pairs: 118419
Train positives: 25404
Validation positives: 8263
Train positive rate: 6.8721%
Validation positive rate: 6.9778%
✅ LightGBM matrices ready


In [24]:
# Cell 18: Save a durable checkpoint of the completed E2.3 feature table,
# labels, exact train/validation split, and LightGBM feature list to disk.

import json
from pathlib import Path

CHECKPOINT_DIR = REPO / "cache" / "model_features"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_CHECKPOINT = (
    CHECKPOINT_DIR / "e23_retrieval_38a0b640f78b_labeled_features.parquet"
)

META_CHECKPOINT = (
    CHECKPOINT_DIR / "e23_retrieval_38a0b640f78b_training_metadata.json"
)

# Save the full 488,090 x 78 feature table, including labels.
features.to_parquet(
    FEATURE_CHECKPOINT,
    index=False,
    compression="snappy",
)

# Save everything needed to recreate the exact matrices/split.
metadata = {
    "run": "e23_retrieval_38a0b640f78b",
    "train_s1_ids": sorted(train_s1_ids),
    "val_s1_ids": sorted(val_s1_ids),
    "model_features": model_features,
    "feature_rows": int(len(features)),
    "feature_columns": int(len(features.columns)),
    "positive_pairs": int(features["label"].sum()),
}

with open(META_CHECKPOINT, "w") as f:
    json.dump(metadata, f, indent=2)

print("Feature checkpoint:")
print(FEATURE_CHECKPOINT)
print(f"Size: {FEATURE_CHECKPOINT.stat().st_size / 1024**2:.1f} MiB")

print("\nMetadata checkpoint:")
print(META_CHECKPOINT)
print(f"Size: {META_CHECKPOINT.stat().st_size / 1024:.1f} KiB")

print("\n✅ CHECKPOINT SAVED — safe to disconnect")

Feature checkpoint:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/model_features/e23_retrieval_38a0b640f78b_labeled_features.parquet
Size: 17.7 MiB

Metadata checkpoint:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/model_features/e23_retrieval_38a0b640f78b_training_metadata.json
Size: 196.6 KiB

✅ CHECKPOINT SAVED — safe to disconnect


In [1]:
# Resume Cell 1: Reload the saved E2.3 checkpoint after the server restart
# and recreate the exact train/validation matrices without recomputing features.

import json
from pathlib import Path
import pandas as pd

REPO = Path.home() / "Amazon-ML-Challenge"

CHECKPOINT_DIR = REPO / "cache" / "model_features"

FEATURE_CHECKPOINT = (
    CHECKPOINT_DIR / "e23_retrieval_38a0b640f78b_labeled_features.parquet"
)

META_CHECKPOINT = (
    CHECKPOINT_DIR / "e23_retrieval_38a0b640f78b_training_metadata.json"
)

features = pd.read_parquet(FEATURE_CHECKPOINT)

with open(META_CHECKPOINT, "r") as f:
    metadata = json.load(f)

train_s1_ids = set(metadata["train_s1_ids"])
val_s1_ids = set(metadata["val_s1_ids"])
model_features = metadata["model_features"]

train_mask = features["source1_entity_id"].isin(train_s1_ids)
val_mask = features["source1_entity_id"].isin(val_s1_ids)

X_train = features.loc[train_mask, model_features]
y_train = features.loc[train_mask, "label"]

X_val = features.loc[val_mask, model_features]
y_val = features.loc[val_mask, "label"]

print("Feature table:", features.shape)
print("Model features:", len(model_features))
print("Train pairs:", len(X_train))
print("Validation pairs:", len(X_val))
print("Train positives:", int(y_train.sum()))
print("Validation positives:", int(y_val.sum()))

print("✅ Checkpoint restored")

Feature table: (488090, 78)
Model features: 75
Train pairs: 369671
Validation pairs: 118419
Train positives: 25404
Validation positives: 8263
✅ Checkpoint restored


In [2]:
# Training Cell 1: Train the E2.3 LightGBM classifier with a live tqdm progress
# bar, validation logging, and early stopping so training progress is visible.

import lightgbm as lgb
import time
from tqdm.auto import tqdm

train_data = lgb.Dataset(
    X_train,
    label=y_train,
    feature_name=model_features,
    free_raw_data=False,
)

val_data = lgb.Dataset(
    X_val,
    label=y_val,
    reference=train_data,
    feature_name=model_features,
    free_raw_data=False,
)

params = {
    "objective": "binary",
    "metric": ["binary_logloss", "auc"],
    "learning_rate": 0.05,
    "num_leaves": 63,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "min_data_in_leaf": 50,
    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,
    "verbosity": -1,
}

NUM_BOOST_ROUND = 2000

progress_bar = tqdm(
    total=NUM_BOOST_ROUND,
    desc="LightGBM boosting",
    unit="round",
    dynamic_ncols=True,
)

last_iteration = {"value": 0}

def tqdm_callback(env):
    current_iteration = env.iteration + 1
    increment = current_iteration - last_iteration["value"]

    if increment > 0:
        progress_bar.update(increment)
        last_iteration["value"] = current_iteration

    # Show current validation metrics in the tqdm bar.
    if env.evaluation_result_list:
        metrics = {}

        for item in env.evaluation_result_list:
            dataset_name = item[0]
            metric_name = item[1]
            metric_value = item[2]

            if dataset_name == "validation":
                metrics[metric_name] = f"{metric_value:.6f}"

        if metrics:
            progress_bar.set_postfix(metrics)

# Run tqdm after metric evaluation each round.
tqdm_callback.order = 20

print(" TRAINING LIGHTGBM NOW")
print(f"Train pairs:      {len(X_train):,}")
print(f"Validation pairs: {len(X_val):,}")
print(f"Features:         {len(model_features)}")
print(f"Max rounds:       {NUM_BOOST_ROUND:,}")
print()

t0 = time.time()

try:
    model = lgb.train(
        params,
        train_data,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[train_data, val_data],
        valid_names=["train", "validation"],
        callbacks=[
            tqdm_callback,
            lgb.log_evaluation(period=25),
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=True,
            ),
        ],
    )

finally:
    progress_bar.close()

elapsed = time.time() - t0

print()
print("TRAINING COMPLETE")
print("Best iteration:", model.best_iteration)
print("Best validation scores:")
print(model.best_score["validation"])
print(f"Training time: {elapsed:.1f}s")

LightGBM boosting:   0%|          | 0/2000 [00:00<?, ?round/s]

 TRAINING LIGHTGBM NOW
Train pairs:      369,671
Validation pairs: 118,419
Features:         75
Max rounds:       2,000

Training until validation scores don't improve for 100 rounds
[25]	train's binary_logloss: 0.0506741	train's auc: 0.999066	validation's binary_logloss: 0.0515806	validation's auc: 0.998696
[50]	train's binary_logloss: 0.0228509	train's auc: 0.999487	validation's binary_logloss: 0.0241725	validation's auc: 0.999101
[75]	train's binary_logloss: 0.0143007	train's auc: 0.999657	validation's binary_logloss: 0.0163697	validation's auc: 0.999285
[100]	train's binary_logloss: 0.0108127	train's auc: 0.999782	validation's binary_logloss: 0.0137894	validation's auc: 0.999341
[125]	train's binary_logloss: 0.00887789	train's auc: 0.999852	validation's binary_logloss: 0.0127566	validation's auc: 0.999392
[150]	train's binary_logloss: 0.00756851	train's auc: 0.9999	validation's binary_logloss: 0.0122733	validation's auc: 0.999445
[175]	train's binary_logloss: 0.00662943	train's auc

In [3]:
# Cell 19: Save the trained E2.3 LightGBM model and its exact 75-feature schema
# so the model can be restored without retraining after a server restart.

import json
from pathlib import Path

MODEL_DIR = REPO / "cache" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = (
    MODEL_DIR / "e23_lgbm_38a0b640f78b.txt"
)

MODEL_META_PATH = (
    MODEL_DIR / "e23_lgbm_38a0b640f78b_metadata.json"
)

model.save_model(
    str(MODEL_PATH),
    num_iteration=model.best_iteration,
)

model_metadata = {
    "run": "e23_retrieval_38a0b640f78b",
    "model_type": "LightGBM",
    "best_iteration": int(model.best_iteration),
    "model_features": model_features,
    "train_pairs": int(len(X_train)),
    "validation_pairs": int(len(X_val)),
    "train_positives": int(y_train.sum()),
    "validation_positives": int(y_val.sum()),
    "validation_binary_logloss": float(
        model.best_score["validation"]["binary_logloss"]
    ),
    "validation_auc": float(
        model.best_score["validation"]["auc"]
    ),
}

with open(MODEL_META_PATH, "w") as f:
    json.dump(model_metadata, f, indent=2)

print("Model saved:")
print(MODEL_PATH)
print(f"Size: {MODEL_PATH.stat().st_size / 1024:.1f} KiB")

print("\nMetadata saved:")
print(MODEL_META_PATH)

print("\n✅ TRAINED LIGHTGBM CHECKPOINTED")

Model saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/models/e23_lgbm_38a0b640f78b.txt
Size: 1772.3 KiB

Metadata saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/models/e23_lgbm_38a0b640f78b_metadata.json

✅ TRAINED LIGHTGBM CHECKPOINTED


In [4]:
# Cell 20: Score all validation candidate pairs with the trained LightGBM model,
# show prediction progress with tqdm, and save the probabilities for threshold tuning.

import numpy as np
from tqdm.auto import tqdm

PRED_CHUNK_SIZE = 25_000

val_scores_parts = []

for start in tqdm(
    range(0, len(X_val), PRED_CHUNK_SIZE),
    desc="Scoring validation pairs",
    unit="chunk",
    dynamic_ncols=True,
):
    end = min(start + PRED_CHUNK_SIZE, len(X_val))

    chunk_scores = model.predict(
        X_val.iloc[start:end],
        num_iteration=model.best_iteration,
    )

    val_scores_parts.append(chunk_scores)

val_scores = np.concatenate(val_scores_parts)

# Attach probabilities back to the validation candidate rows.
val_results = features.loc[
    val_mask,
    ["source1_entity_id", "candidate_entity_id", "label"]
].copy()

val_results["score"] = val_scores.astype("float32")

PRED_PATH = (
    REPO
    / "cache"
    / "predictions"
    / "e23_lgbm_38a0b640f78b_validation.parquet"
)

PRED_PATH.parent.mkdir(parents=True, exist_ok=True)

val_results.to_parquet(
    PRED_PATH,
    index=False,
    compression="snappy",
)

print("Validation predictions:", val_results.shape)
print("Positive labels:", int(val_results["label"].sum()))
print("Score min:", float(val_results["score"].min()))
print("Score max:", float(val_results["score"].max()))
print("Score mean:", float(val_results["score"].mean()))
print("\nSaved to:")
print(PRED_PATH)

print("\n✅ Validation probabilities generated and checkpointed")

Scoring validation pairs:   0%|          | 0/5 [00:00<?, ?chunk/s]

Validation predictions: (118419, 4)
Positive labels: 8263
Score min: 2.2736253413313534e-06
Score max: 0.9999555349349976
Score mean: 0.06967224180698395

Saved to:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/predictions/e23_lgbm_38a0b640f78b_validation.parquet

✅ Validation probabilities generated and checkpointed


In [5]:
# Cell 21: Sweep decision thresholds using the true competition macro-F0.5 metric.
# This includes retrieval-missed ground-truth links, not just labels present in candidates.

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

BETA = 0.5
BETA2 = BETA ** 2

# Reload the complete training ground truth.
GT_PATH = (
    REPO
    / "student_resource"
    / "dataset"
    / "train"
    / "train_ground_truth.tsv"
)

gt = pd.read_csv(GT_PATH, sep="\t")

# Keep only the exact validation Source1 entities.
val_gt = gt[
    gt["source1_entity_id"].isin(val_s1_ids)
].copy()

# Count total TRUE links for every validation entity.
# This deliberately includes links that retrieval failed to generate as candidates.
val_gt["truth_count"] = (
    val_gt["matched_entity_ids"]
    .fillna("")
    .astype(str)
    .apply(lambda x: 0 if x.strip() == "" else len(x.split(",")))
)

truth_count_map = dict(
    zip(val_gt["source1_entity_id"], val_gt["truth_count"])
)

# Encode validation entities as integer indices for very fast bincount operations.
val_entities = sorted(val_s1_ids)
entity_to_idx = {
    entity_id: i
    for i, entity_id in enumerate(val_entities)
}

entity_idx = (
    val_results["source1_entity_id"]
    .map(entity_to_idx)
    .to_numpy(dtype=np.int32)
)

scores = val_results["score"].to_numpy(dtype=np.float32)
labels = val_results["label"].to_numpy(dtype=np.uint8)

total_truth = np.array(
    [truth_count_map.get(entity_id, 0) for entity_id in val_entities],
    dtype=np.int32,
)

n_entities = len(val_entities)

# Dense threshold sweep.
thresholds = np.linspace(0.01, 0.995, 500)

results = []

for threshold in tqdm(
    thresholds,
    desc="Sweeping F0.5 thresholds",
    unit="threshold",
    dynamic_ncols=True,
):
    predicted = scores >= threshold

    pred_count = np.bincount(
        entity_idx[predicted],
        minlength=n_entities,
    )

    tp = np.bincount(
        entity_idx[predicted & (labels == 1)],
        minlength=n_entities,
    )

    fp = pred_count - tp
    fn = total_truth - tp

    denominator = (
        (1 + BETA2) * tp
        + BETA2 * fn
        + fp
    )

    # Official special case:
    # empty ground truth + empty prediction => score 1.
    entity_f05 = np.zeros(n_entities, dtype=np.float64)

    both_empty = (total_truth == 0) & (pred_count == 0)
    entity_f05[both_empty] = 1.0

    normal = denominator > 0

    entity_f05[normal] = (
        (1 + BETA2) * tp[normal]
        / denominator[normal]
    )

    results.append(
        (
            float(threshold),
            float(entity_f05.mean()),
            int(tp.sum()),
            int(fp.sum()),
            int(fn.sum()),
        )
    )

threshold_results = pd.DataFrame(
    results,
    columns=[
        "threshold",
        "macro_f0.5",
        "tp",
        "fp",
        "fn",
    ],
)

best_row = threshold_results.loc[
    threshold_results["macro_f0.5"].idxmax()
]

print("\n🏆 BEST VALIDATION THRESHOLD")
print(f"Threshold:  {best_row['threshold']:.6f}")
print(f"Macro F0.5: {best_row['macro_f0.5']:.6f}")
print(f"TP:         {int(best_row['tp']):,}")
print(f"FP:         {int(best_row['fp']):,}")
print(f"FN:         {int(best_row['fn']):,}")

print("\nTop 10 thresholds:")
display(
    threshold_results
    .sort_values("macro_f0.5", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

Sweeping F0.5 thresholds:   0%|          | 0/500 [00:00<?, ?threshold/s]


🏆 BEST VALIDATION THRESHOLD
Threshold:  0.710752
Macro F0.5: 0.962594
TP:         7,887
FP:         143
FN:         582

Top 10 thresholds:


,threshold,macro_f0.5,tp,fp,fn
0,0.710752,0.962594,7887,143,582
1,0.714699,0.962529,7884,142,585
2,0.742335,0.962514,7854,131,615
3,0.722595,0.962508,7876,140,593
4,0.712725,0.962480,7884,143,585
5,0.623898,0.962469,7944,168,525
6,0.616002,0.962460,7950,171,519
7,0.716673,0.962454,7882,142,587
8,0.720621,0.962448,7878,141,591
9,0.612054,0.962442,7951,172,518


In [6]:
# Cell 22: Evaluate the best E2.3 threshold separately for India and the US
# to identify where the remaining validation errors and F0.5 loss come from.

BEST_THRESHOLD = float(best_row["threshold"])

# Reload the normalized 10k Source1 table to obtain each entity's country.
SOURCE1_PATH = (
    REPO
    / "cache"
    / "normalized"
    / "e21_lgbm_c449c0c6b281_source1.parquet"
)

source1_eval = pd.read_parquet(
    SOURCE1_PATH,
    columns=["entity_id", "country"],
)

country_map = dict(
    zip(source1_eval["entity_id"], source1_eval["country"])
)

entity_country = np.array(
    [country_map[eid] for eid in val_entities]
)

predicted = scores >= BEST_THRESHOLD

pred_count = np.bincount(
    entity_idx[predicted],
    minlength=n_entities,
)

tp = np.bincount(
    entity_idx[predicted & (labels == 1)],
    minlength=n_entities,
)

fp = pred_count - tp
fn = total_truth - tp

denominator = (
    (1 + BETA2) * tp
    + BETA2 * fn
    + fp
)

entity_f05 = np.zeros(n_entities, dtype=np.float64)

both_empty = (total_truth == 0) & (pred_count == 0)
entity_f05[both_empty] = 1.0

normal = denominator > 0
entity_f05[normal] = (
    (1 + BETA2) * tp[normal]
    / denominator[normal]
)

rows = []

for country in sorted(set(entity_country)):
    mask = entity_country == country

    rows.append({
        "country": country,
        "entities": int(mask.sum()),
        "macro_f0.5": float(entity_f05[mask].mean()),
        "tp": int(tp[mask].sum()),
        "fp": int(fp[mask].sum()),
        "fn": int(fn[mask].sum()),
        "truth_links": int(total_truth[mask].sum()),
    })

country_results = pd.DataFrame(rows)

print(f"Threshold: {BEST_THRESHOLD:.6f}")
display(country_results)

print(
    "\nOverall macro F0.5:",
    f"{entity_f05.mean():.6f}"
)

Threshold: 0.710752


,country,entities,macro_f0.5,tp,fp,fn,truth_links
0,India,953,0.945960,2979,70,325,3304
1,US,1479,0.973312,4908,73,257,5165



Overall macro F0.5: 0.962594


In [7]:
# Cell 23: Decompose validation false negatives into retrieval misses versus
# classifier rejections, separately for India and the US.

BEST_THRESHOLD = float(best_row["threshold"])

# True candidate links actually recovered by retrieval.
recovered_true = val_results[val_results["label"] == 1].copy()

# Of the recovered true links, these were rejected by LightGBM.
classifier_rejected = recovered_true[
    recovered_true["score"] < BEST_THRESHOLD
].copy()

# Count recovered true links by Source1 entity.
recovered_true_count = (
    recovered_true
    .groupby("source1_entity_id")
    .size()
    .to_dict()
)

# Count classifier-rejected true links by Source1 entity.
classifier_rejected_count = (
    classifier_rejected
    .groupby("source1_entity_id")
    .size()
    .to_dict()
)

error_rows = []

for entity_id in val_entities:
    country = country_map[entity_id]

    truth = int(truth_count_map.get(entity_id, 0))
    recovered = int(recovered_true_count.get(entity_id, 0))
    rejected = int(classifier_rejected_count.get(entity_id, 0))

    retrieval_missed = truth - recovered

    error_rows.append({
        "source1_entity_id": entity_id,
        "country": country,
        "truth_links": truth,
        "retrieved_true_links": recovered,
        "retrieval_misses": retrieval_missed,
        "classifier_rejections": rejected,
    })

error_breakdown = pd.DataFrame(error_rows)

country_error_summary = (
    error_breakdown
    .groupby("country", as_index=False)
    .agg(
        entities=("source1_entity_id", "count"),
        truth_links=("truth_links", "sum"),
        retrieved_true_links=("retrieved_true_links", "sum"),
        retrieval_misses=("retrieval_misses", "sum"),
        classifier_rejections=("classifier_rejections", "sum"),
    )
)

country_error_summary["retrieval_miss_rate"] = (
    country_error_summary["retrieval_misses"]
    / country_error_summary["truth_links"]
)

country_error_summary["classifier_reject_rate_of_retrieved"] = (
    country_error_summary["classifier_rejections"]
    / country_error_summary["retrieved_true_links"]
)

display(country_error_summary)

print("\nTotal FN decomposition:")
print(
    "Retrieval misses:",
    int(country_error_summary["retrieval_misses"].sum())
)
print(
    "Classifier rejections:",
    int(country_error_summary["classifier_rejections"].sum())
)

,country,entities,truth_links,retrieved_true_links,retrieval_misses,classifier_rejections,retrieval_miss_rate,classifier_reject_rate_of_retrieved
0,India,953,3304,3164,140,185,0.042373,0.058470
1,US,1479,5165,5099,66,191,0.012778,0.037458



Total FN decomposition:
Retrieval misses: 206
Classifier rejections: 376


In [8]:
# Cell 24: Sweep separate LightGBM decision thresholds for India and the US
# to test whether country-specific thresholds improve macro-F0.5 over one global threshold.

country_threshold_results = {}

for country in ["India", "US"]:
    country_entity_mask = entity_country == country
    country_entity_indices = np.where(country_entity_mask)[0]

    # Map global entity indices -> compact country-local indices.
    global_to_local = {
        global_idx: local_idx
        for local_idx, global_idx in enumerate(country_entity_indices)
    }

    pair_mask = np.isin(entity_idx, country_entity_indices)

    country_scores = scores[pair_mask]
    country_labels = labels[pair_mask]
    country_global_idx = entity_idx[pair_mask]

    country_local_idx = np.array(
        [global_to_local[i] for i in country_global_idx],
        dtype=np.int32,
    )

    country_truth = total_truth[country_entity_indices]
    n_country_entities = len(country_entity_indices)

    rows = []

    for threshold in tqdm(
        thresholds,
        desc=f"Sweeping {country}",
        unit="threshold",
        dynamic_ncols=True,
    ):
        predicted = country_scores >= threshold

        pred_count = np.bincount(
            country_local_idx[predicted],
            minlength=n_country_entities,
        )

        tp_country = np.bincount(
            country_local_idx[predicted & (country_labels == 1)],
            minlength=n_country_entities,
        )

        fp_country = pred_count - tp_country
        fn_country = country_truth - tp_country

        denominator = (
            (1 + BETA2) * tp_country
            + BETA2 * fn_country
            + fp_country
        )

        entity_scores = np.zeros(
            n_country_entities,
            dtype=np.float64,
        )

        both_empty = (
            (country_truth == 0)
            & (pred_count == 0)
        )
        entity_scores[both_empty] = 1.0

        normal = denominator > 0
        entity_scores[normal] = (
            (1 + BETA2) * tp_country[normal]
            / denominator[normal]
        )

        rows.append({
            "threshold": float(threshold),
            "macro_f0.5": float(entity_scores.mean()),
            "tp": int(tp_country.sum()),
            "fp": int(fp_country.sum()),
            "fn": int(fn_country.sum()),
        })

    result = pd.DataFrame(rows)
    country_threshold_results[country] = result

    best = result.loc[result["macro_f0.5"].idxmax()]

    print(f"\n🏆 {country}")
    print(f"Best threshold: {best['threshold']:.6f}")
    print(f"Macro F0.5:     {best['macro_f0.5']:.6f}")
    print(f"TP:             {int(best['tp']):,}")
    print(f"FP:             {int(best['fp']):,}")
    print(f"FN:             {int(best['fn']):,}")

    display(
        result.sort_values(
            "macro_f0.5",
            ascending=False,
        ).head(5)
    )

Sweeping India:   0%|          | 0/500 [00:00<?, ?threshold/s]


🏆 India
Best threshold: 0.616002
Macro F0.5:     0.946879
TP:             3,012
FP:             84
FN:             292


,threshold,macro_f0.5,tp,fp,fn
307,0.616002,0.946879,3012,84,292
304,0.610080,0.946833,3013,85,291
308,0.617976,0.946760,3011,84,293
310,0.621924,0.946705,3010,83,294
311,0.623898,0.946705,3010,83,294


Sweeping US:   0%|          | 0/500 [00:00<?, ?threshold/s]


🏆 US
Best threshold: 0.742335
Macro F0.5:     0.973603
TP:             4,885
FP:             65
FN:             280


,threshold,macro_f0.5,tp,fp,fn
371,0.742335,0.973603,4885,65,280
368,0.736413,0.973589,4889,68,276
367,0.734439,0.973538,4891,70,274
369,0.738387,0.973387,4887,68,278
370,0.740361,0.973387,4887,68,278


In [9]:
# Cell 25: Apply the best country-specific thresholds simultaneously and compute
# the combined validation macro-F0.5 to see the gain over the global threshold.

COUNTRY_THRESHOLDS = {
    "India": 0.616002,
    "US": 0.742335,
}

pair_countries = np.array(
    [country_map[eid] for eid in val_results["source1_entity_id"]]
)

pair_thresholds = np.array(
    [COUNTRY_THRESHOLDS[c] for c in pair_countries],
    dtype=np.float32,
)

predicted = scores >= pair_thresholds

pred_count = np.bincount(
    entity_idx[predicted],
    minlength=n_entities,
)

tp_country_specific = np.bincount(
    entity_idx[predicted & (labels == 1)],
    minlength=n_entities,
)

fp_country_specific = pred_count - tp_country_specific
fn_country_specific = total_truth - tp_country_specific

denominator = (
    (1 + BETA2) * tp_country_specific
    + BETA2 * fn_country_specific
    + fp_country_specific
)

entity_f05_country_specific = np.zeros(
    n_entities,
    dtype=np.float64,
)

both_empty = (
    (total_truth == 0)
    & (pred_count == 0)
)

entity_f05_country_specific[both_empty] = 1.0

normal = denominator > 0

entity_f05_country_specific[normal] = (
    (1 + BETA2) * tp_country_specific[normal]
    / denominator[normal]
)

combined_score = entity_f05_country_specific.mean()

print("Country-specific thresholds:")
print(COUNTRY_THRESHOLDS)

print("\nCombined validation results:")
print(f"Macro F0.5: {combined_score:.6f}")
print(f"TP:         {int(tp_country_specific.sum()):,}")
print(f"FP:         {int(fp_country_specific.sum()):,}")
print(f"FN:         {int(fn_country_specific.sum()):,}")

print("\nComparison:")
print(f"Global threshold score:   {best_row['macro_f0.5']:.6f}")
print(f"Country-specific score:   {combined_score:.6f}")
print(
    f"Absolute improvement:     "
    f"{combined_score - best_row['macro_f0.5']:+.6f}"
)

Country-specific thresholds:
{'India': 0.616002, 'US': 0.742335}

Combined validation results:
Macro F0.5: 0.963131
TP:         7,897
FP:         149
FN:         572

Comparison:
Global threshold score:   0.962594
Country-specific score:   0.963131
Absolute improvement:     +0.000537


In [10]:
# Cell 26: Save the selected country-specific thresholds and validation metrics
# so inference can reproduce the exact decoder without re-running threshold sweeps.

import json

DECODER_PATH = (
    REPO
    / "cache"
    / "models"
    / "e23_lgbm_38a0b640f78b_decoder.json"
)

decoder_config = {
    "model_run": "e23_lgbm_38a0b640f78b",
    "metric": "macro_f0.5",
    "beta": 0.5,

    "global_threshold": float(best_row["threshold"]),
    "global_validation_macro_f0.5": float(
        best_row["macro_f0.5"]
    ),

    "country_thresholds": {
        "India": 0.616002,
        "US": 0.742335,
    },

    "country_specific_validation_macro_f0.5": float(
        combined_score
    ),

    "validation_tp": int(tp_country_specific.sum()),
    "validation_fp": int(fp_country_specific.sum()),
    "validation_fn": int(fn_country_specific.sum()),

    # France is unseen in training; this is intentionally not finalized yet.
    "france_threshold": None,
}

with open(DECODER_PATH, "w") as f:
    json.dump(decoder_config, f, indent=2)

print("Decoder config saved:")
print(DECODER_PATH)

print("\nSaved thresholds:")
print("India:", decoder_config["country_thresholds"]["India"])
print("US:   ", decoder_config["country_thresholds"]["US"])

print(
    "\nValidation macro F0.5:",
    f"{decoder_config['country_specific_validation_macro_f0.5']:.6f}",
)

print("\n E2.3 decoder checkpointed")

Decoder config saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/cache/models/e23_lgbm_38a0b640f78b_decoder.json

Saved thresholds:
India: 0.616002
US:    0.742335

Validation macro F0.5: 0.963131

 E2.3 decoder checkpointed


In [11]:
# Cell 27: Inspect LightGBM feature importance to see which legacy and E2.3
# address-retrieval features contribute most to the trained classifier.

importance = pd.DataFrame({
    "feature": model.feature_name(),
    "gain": model.feature_importance(
        importance_type="gain",
        iteration=model.best_iteration,
    ),
    "split": model.feature_importance(
        importance_type="split",
        iteration=model.best_iteration,
    ),
})

importance["gain_pct"] = (
    100 * importance["gain"] / importance["gain"].sum()
)

importance = importance.sort_values(
    "gain",
    ascending=False,
).reset_index(drop=True)

print("Top 25 features by gain:")
display(
    importance[
        ["feature", "gain_pct", "split"]
    ].head(25)
)

e23_feature_set = set(E23_MODEL_FEATURES)

e23_importance = importance[
    importance["feature"].isin(e23_feature_set)
].copy()

print(
    "\nTotal gain from E2.3-specific features:",
    f"{e23_importance['gain_pct'].sum():.2f}%"
)

print("\nE2.3-specific feature importance:")
display(
    e23_importance[
        ["feature", "gain_pct", "split"]
    ].reset_index(drop=True)
)

Top 25 features by gain:


,feature,gain_pct,split
0,address_tfidf_similarity,43.014107,560
1,retrieval_channel_count,11.959685,37
2,relative_address_tfidf_rank,7.637506,318
3,fusion_score,7.480720,493
4,address_token_jaccard,5.500558,872
5,name_character_similarity,5.334405,1241
6,name_token_jaccard,3.221939,916
7,address_length_ratio,2.145782,788
8,numeric_token_conflict,1.933664,232
9,numeric_token_overlap,1.914968,401


NameError: name 'E23_MODEL_FEATURES' is not defined

In [12]:
# Cell 27B: Recreate the E2.3-specific feature list after the kernel restart
# and measure how much total LightGBM gain comes from the new retrieval features.

E23_MODEL_FEATURES = [
    "address_tfidf_similarity",
    "address_tfidf_rank",
    "address_route_postcode",
    "address_route_postcode_prefix",
    "address_route_numeric",
    "address_route_rare_token",
    "address_route_country_fallback",
    "address_route_count",
    "address_preblock_pool_size",
    "has_address_tfidf",
    "block_address_tfidf",
    "is_address_tfidf_only",
    "retrieval_channel_count",
    "relative_address_tfidf_rank",
    "address_tfidf_gap_to_best",
]

e23_importance = importance[
    importance["feature"].isin(E23_MODEL_FEATURES)
].copy()

print(
    "Total gain from E2.3-specific features:",
    f"{e23_importance['gain_pct'].sum():.2f}%"
)

display(
    e23_importance[
        ["feature", "gain_pct", "split"]
    ].reset_index(drop=True)
)

Total gain from E2.3-specific features: 63.51%


,feature,gain_pct,split
0,address_tfidf_similarity,43.014107,560
1,retrieval_channel_count,11.959685,37
2,relative_address_tfidf_rank,7.637506,318
3,address_route_count,0.316272,88
4,address_tfidf_gap_to_best,0.263988,597
5,address_preblock_pool_size,0.214910,481
6,address_tfidf_rank,0.068240,119
7,address_route_rare_token,0.014583,25
8,address_route_numeric,0.010419,27
9,address_route_postcode,0.006917,21


In [13]:
# Cell 28: Save LightGBM feature importance and the E2.3 gain summary so the
# experiment record survives notebook/server restarts and can be compared later.

IMPORTANCE_PATH = (
    REPO
    / "experiments"
    / "e23_lgbm_38a0b640f78b_feature_importance.csv"
)

SUMMARY_PATH = (
    REPO
    / "experiments"
    / "e23_lgbm_38a0b640f78b_summary.json"
)

IMPORTANCE_PATH.parent.mkdir(parents=True, exist_ok=True)

importance.to_csv(
    IMPORTANCE_PATH,
    index=False,
)

experiment_summary = {
    "run": "e23_lgbm_38a0b640f78b",
    "best_iteration": int(model.best_iteration),
    "validation_auc": float(
        model.best_score["validation"]["auc"]
    ),
    "validation_logloss": float(
        model.best_score["validation"]["binary_logloss"]
    ),
    "global_threshold": float(best_row["threshold"]),
    "global_macro_f0.5": float(best_row["macro_f0.5"]),
    "country_thresholds": {
        "India": 0.616002,
        "US": 0.742335,
    },
    "country_specific_macro_f0.5": float(combined_score),
    "e23_feature_gain_pct": float(
        e23_importance["gain_pct"].sum()
    ),
    "top_feature": str(importance.iloc[0]["feature"]),
    "top_feature_gain_pct": float(
        importance.iloc[0]["gain_pct"]
    ),
}

with open(SUMMARY_PATH, "w") as f:
    json.dump(
        experiment_summary,
        f,
        indent=2,
    )

print("Feature importance saved:")
print(IMPORTANCE_PATH)

print("\nExperiment summary saved:")
print(SUMMARY_PATH)

print("\nKey result:")
print(
    f"E2.3-specific feature gain: "
    f"{experiment_summary['e23_feature_gain_pct']:.2f}%"
)
print(
    f"Best validation macro F0.5: "
    f"{experiment_summary['country_specific_macro_f0.5']:.6f}"
)

print("\n✅ E2.3 model evaluation checkpointed")

Feature importance saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/experiments/e23_lgbm_38a0b640f78b_feature_importance.csv

Experiment summary saved:
/home/jupyter-1nt23cb058/Amazon-ML-Challenge/experiments/e23_lgbm_38a0b640f78b_summary.json

Key result:
E2.3-specific feature gain: 63.51%
Best validation macro F0.5: 0.963131

✅ E2.3 model evaluation checkpointed
